# CSC 4792 Group 40: Kafue Town Council

This notebook documents the acquisition, cleaning, validation and export of the Group 40 public-information dataset. The assigned organisation is Kafue Town Council.

The primary source is the official council website and its linked public PDF/XLSX resources. Kaggle is used as the distribution location, not as the evidence source. Personal identifiers from source workbooks (NRC numbers, guardian names and phone numbers) are intentionally excluded.

## 1. Reproducible acquisition

Official website: https://www.kafuecouncil.gov.zm/

The council publications page is https://www.kafuecouncil.gov.zm/?page_id=195. The source manifest records the downloaded filenames and this official page. To refresh the release, download the linked public files into `source_docs/`, run `python build_enriched_data.py`, and then rerun the checks below. The raw source documents are not committed because the submission should distribute the derived CSVs with their traceability fields.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_URL = 'https://www.kafuecouncil.gov.zm/'
DATA_DIR = Path('.')
CSV_FILES = sorted(DATA_DIR.glob('db-unza26-csc4792-*.csv'))
print([p.name for p in CSV_FILES])

## 2. Load the released tables

All released tables use the required pipe (`|`) separator. Each table has its own documented schema because project, performance, finance and document-page records have different fields.

In [ ]:
def load_pipe(name):
    path = DATA_DIR / name
    return pd.read_csv(path, sep='|', dtype=str, keep_default_na=False)

tables = {p.stem: load_pipe(p.name) for p in CSV_FILES}
for name, table in tables.items():
    print(f'{name}: {table.shape[0]:,} rows x {table.shape[1]} columns')

In [ ]:
cdf = tables['db-unza26-csc4792-kafue_cdf_records']
performance = tables['db-unza26-csc4792-kafue_performance_records']
finance = tables['db-unza26-csc4792-kafue_finance_records']
pages = tables['db-unza26-csc4792-kafue_document_pages']
council = tables['db-unza26-csc4792-kafue_council_information']
manifest = tables['db-unza26-csc4792-source_manifest']
cdf['category'].value_counts()

## 3. Quality and privacy checks

The checks below verify the assignment format, stable IDs, source traceability, and the exclusion of obvious personal-identifier fields. Blank values are retained as blank because the source may not report a value.

In [ ]:
all_tables = list(tables.values())
for path in CSV_FILES:
    assert path.name.startswith('db-unza26-csc4792-') and path.suffix == '.csv'
    check = pd.read_csv(path, sep='|', dtype=str, keep_default_na=False)
    assert len(check.columns) > 1

for table in [cdf, performance, finance, pages, council]:
    assert table['record_id'].is_unique
    assert (table['source_url'].str.len() > 0).all()

for forbidden in ['nrc', 'mobile', 'phone', 'guardian', 'parent']:
    assert not any(forbidden in col.lower() for table in all_tables for col in table.columns)

print('All format, traceability, uniqueness and privacy checks passed.')

In [ ]:
quality = pd.DataFrame({
    'table': ['council', 'cdf', 'performance', 'finance', 'document_pages', 'source_manifest'],
    'rows': [len(council), len(cdf), len(performance), len(finance), len(pages), len(manifest)],
    'unique_ids': [council.record_id.nunique(), cdf.record_id.nunique(), performance.record_id.nunique(), finance.record_id.nunique(), pages.record_id.nunique(), manifest.source_file.nunique()]
})
quality

## 4. Example analysis views

These views show how the release can support data-mining questions without changing the source values. Amounts are kept as reported; users should confirm currency and column meaning in the source workbook before financial modelling.

In [ ]:
project_summary = (cdf[cdf['category'].eq('community_project')]
                  .groupby(['source_year', 'sector'], dropna=False)
                  .size().reset_index(name='records')
                  .sort_values(['source_year', 'records'], ascending=[True, False]))
project_summary.head(20)

In [ ]:
performance[['source_sheet', 'programme', 'key_output', 'annual_target', 'achieved_output', 'variance']].head(10)

## 5. Export and limitations

The generated files are already pipe-delimited and named with the required `db-unza26-csc4792-[DESCRIPTION].csv` convention. The release is source-traceable but not a claim that every council PDF is machine-readable: some documents contain scanned pages, so document-page rows may have blank extracted text while still preserving the file and page reference.

In [ ]:
# Optional re-export after a documented cleaning change.
# cdf.to_csv('db-unza26-csc4792-kafue_cdf_records.csv', sep='|', index=False)
print('Notebook validation complete. Preserve the source_url, source_file and source_sheet fields when editing.')